# Robust Image Preprocessing, Harmonization & Balancing Pipeline for 6-Dataset DR Hybrid Pool

**Author:** Expert Computer Vision & Medical AI Assistant  
**Dataset Scope:** APTOS 2019, EyePACS, Messidor-2, IDRiD, DeepDRiD, DDR (53,977 raw images).  
**Target Outputs:** Dual resolutions (`512x512`), 70/15/15 leak-free stratified splits, CLAHE contrast enhancement, and targeted training set class balancing (exactly 2,000 images/class = 10,000 total train images).



In [27]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import albumentations as A
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
# Suppress non-critical warnings
warnings.filterwarnings("ignore")

# Random seeds for research reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"[INFO] Initialized preprocessing pipeline with SEED={SEED}")



[INFO] Initialized preprocessing pipeline with SEED=42


## 1. Dynamic Path Resolution & Configuration Dictionary



In [28]:
def get_base_paths() -> tuple[str, str]:
    """
    Dynamically resolves relative paths to `data/raw/` and `data/processed/`
    based on current working directory.
    """
    cwd = os.getcwd()
    raw_dir = os.path.abspath(os.path.join(cwd, "..", "..", "..", "..", "data", "raw"))
    proc_dir = os.path.abspath(os.path.join(cwd, "..", "..", "..", "..", "data", "processed"))
    os.makedirs(proc_dir, exist_ok=True)
    return raw_dir, proc_dir

BASE_RAW_DIR, BASE_PROCESSED_DIR = get_base_paths()
HYBRID_RAW_REF = os.path.join(BASE_RAW_DIR, "hybrid")
NATIVE_PROCESSED_DIR = os.path.join(HYBRID_RAW_REF, "native_cropped")
os.makedirs(HYBRID_RAW_REF, exist_ok=True)
os.makedirs(NATIVE_PROCESSED_DIR, exist_ok=True)
print(f"[INFO] Raw Dir: {BASE_RAW_DIR}")
# Raw dataset configuration dictionary
DATASET_CONFIG = {
    "APTOS": {
        "name": "APTOS 2019",
        "csv_path": os.path.join("aptos", "aptos_total.csv"),
        "img_dir": os.path.join("aptos", "aptos_total_images"),
        "id_col": "id_code",
        "label_col": "diagnosis",
        "id_suffix": ".png"
    },
    "DDR": {
        "name": "DDR Dataset",
        "csv_path": os.path.join("ddr", "DR_grading.csv"),
        "img_dir": os.path.join("ddr", "DR_grading", "DR_grading"),
        "id_col": "id_code",
        "label_col": "diagnosis",
        "id_suffix": ".jpg"
    },
    "DeepDRiD": {
        "name": "DeepDRiD",
        "csv_path": os.path.join("deepdrid", "deepdrid_total.csv"),
        "img_dir": os.path.join("deepdrid", "deepdrid_total_images"),
        "id_col": "image_id",
        "label_col": "patient_DR_Level",
        "id_suffix": ".jpg"
    },
    "IDRiD": {
        "name": "IDRiD",
        "csv_path": os.path.join("idrid", "idrid_total.csv"),
        "img_dir": os.path.join("idrid", "idrid_total_images"),
        "id_col": "image_id",
        "label_col": "diagnosis",
        "id_suffix": ".jpg"
    },
    "Messidor-2": {
        "name": "Messidor-2",
        "csv_path": os.path.join("messidor2preprocess", "messidor_data.csv"),
        "img_dir": os.path.join("messidor2preprocess", "messidor-2", "messidor-2", "preprocess"),
        "id_col": "id_code",
        "label_col": "diagnosis",
        "id_suffix": ".png"
    },
    "EyePACS": {
        "name": "EyePACS (Kaggle)",
        "csv_path": os.path.join("zipEyepacs", "trainLabels.csv", "trainLabels.csv"),
        "img_dir": os.path.join("zipEyepacs", "train"),
        "id_col": "image",
        "label_col": "level",
        "id_suffix": ".jpeg"
    }
}

[INFO] Raw Dir: d:\Github\deep-learning-for-computer-vision\data\raw


## 2. Step 1: Master Dataset Harmonization & Schema Standardization

**Clinical Justification:** All 6 datasets map to the International Clinical Diabetic Retinopathy Severity Scale (ICDRS 0-4).
Merging multi-center datasets exposes the network to diverse camera sensors (Topcon, Canon, Zeiss) and illumination conditions, improving cross-domain generalizability.



In [29]:
def parse_patient_and_side(image_id: str, dataset_key: str) -> tuple[str, str]:
    """
    Trích xuất chính xác patient_id và mắt (left/right) từ tên ảnh của từng dataset cụ thể.
    Đảm bảo mắt trái và mắt phải của CÙNG MỘT BỆNH NHÂN luôn có chung một patient_id.
    Gắn prefix dataset_key:: để chống va chạm mã bệnh nhân giữa các bộ dữ liệu.
    """
    img_str = str(image_id).lower()
    k = dataset_key.lower()
    
    if "idrid" in k:
        # IDRiD: 'train_IDRiD_001' hoặc 'IDRiD_001' -> lấy phần sau gạch dưới cuối ("001")
        base = img_str.replace(".jpg", "").replace(".png", "")
        pid = base.split("_")[-1]
    elif "eyepacs" in k or "deepdrid" in k:
        # EyePACS: '10_left' -> '10'; DeepDRiD: '104_l1' -> '104'
        pid = img_str.split("_")[0]
    elif "ddr" in k:
        # DDR dùng gạch ngang: 007-0004-000 -> nhóm theo 2 trường đầu
        parts = img_str.replace(".jpg", "").replace(".png", "").split("-")
        pid = "-".join(parts[:2]) if len(parts) >= 2 else img_str.split(".")[0]
    elif "messidor" in k:
        # Messidor-2 có cặp 2 mắt: 20051020_43808_0100_PP -> nhóm theo ngày khám (trường đầu)
        pid = img_str.split("_")[0]
    else:
        # APTOS: 1 ảnh = 1 bệnh nhân
        pid = img_str
        
    side = "left" if "left" in img_str or "_l" in img_str else \
           "right" if "right" in img_str or "_r" in img_str else "unknown"
    return f"{dataset_key}::{pid}", side

def load_and_standardize_single_dataset(key: str, cfg: dict, base_dir: str) -> pd.DataFrame:
    full_csv_path = os.path.join(base_dir, cfg["csv_path"])
    full_img_dir = os.path.join(base_dir, cfg["img_dir"])
    
    if not os.path.exists(full_csv_path):
        print(f"[WARNING] Skipping {key} - CSV not found: {full_csv_path}")
        return pd.DataFrame()
        
    raw_df = pd.read_csv(full_csv_path)
    label_col = cfg["label_col"]
    if label_col not in raw_df.columns:
        possible = [c for c in raw_df.columns if "dr" in c.lower() or "level" in c.lower() or "diagnosis" in c.lower()]
        if possible:
            label_col = possible[0]
            
    df = raw_df[[cfg["id_col"], label_col]].copy()
    df.columns = ["image_id", "diagnosis"]
    df["dataset_name"] = cfg["name"]
    df["dataset_key"] = key
    
    # Ensure correct suffix
    if cfg["id_suffix"]:
        df["image_file"] = df["image_id"].astype(str).apply(lambda x: x if x.endswith(cfg["id_suffix"]) else x + cfg["id_suffix"])
    else:
        df["image_file"] = df["image_id"].astype(str)
        
    df["raw_full_path"] = df["image_file"].apply(lambda f: os.path.join(full_img_dir, f))
    
    # Clean non-integer labels
    df["diagnosis"] = pd.to_numeric(df["diagnosis"], errors="coerce")
    df = df.dropna(subset=["diagnosis"]).copy()
    df["diagnosis"] = df["diagnosis"].astype(int)
    df = df[df["diagnosis"].isin([0, 1, 2, 3, 4])]
    
    # GỌI HÀM PARSE ĐỂ TẠO PATIENT_ID CHỐNG LEAKAGE
    parsed_metadata = df["image_id"].apply(lambda x: parse_patient_and_side(x, dataset_key=key))
    df["patient_id"] = [p[0] for p in parsed_metadata]
    df["side"] = [p[1] for p in parsed_metadata]
    return df

def merge_and_harmonize_all_datasets(config_dict: dict, base_dir: str) -> pd.DataFrame:
    dfs = []
    for key, cfg in config_dict.items():
        df_sub = load_and_standardize_single_dataset(key, cfg, base_dir)
        if not df_sub.empty:
            dfs.append(df_sub)
            print(f"[HARMONIZED] {cfg['name']}: {len(df_sub):,} records")
            
    master_df = pd.concat(dfs, ignore_index=True)
    print(f"\n[SUCCESS] Master Harmonized Dataset: {len(master_df):,} total images across {master_df['dataset_name'].nunique()} sources.")
    return master_df

master_df = merge_and_harmonize_all_datasets(DATASET_CONFIG, BASE_RAW_DIR)



[HARMONIZED] APTOS 2019: 3,662 records
[HARMONIZED] DDR Dataset: 12,522 records
[HARMONIZED] DeepDRiD: 2,000 records
[HARMONIZED] IDRiD: 516 records
[HARMONIZED] Messidor-2: 1,744 records
[HARMONIZED] EyePACS (Kaggle): 35,126 records

[SUCCESS] Master Harmonized Dataset: 55,570 total images across 6 sources.


In [30]:
# Kiểm tra tính toàn vẹn và số lượng bệnh nhân duy nhất (IDRiD phải ra ~400+, không phải 1)
print("=== KIỂM TOÁN SỐ LƯỢNG BỆNH NHÂN DUY NHẤT THEO DATASET ===")
patient_counts = master_df.groupby("dataset_name")["patient_id"].nunique()
print(patient_counts)
print("\nTổng số bệnh nhân duy nhất toàn bộ cohort:", master_df["patient_id"].nunique())
master_df

=== KIỂM TOÁN SỐ LƯỢNG BỆNH NHÂN DUY NHẤT THEO DATASET ===
dataset_name
APTOS 2019           3662
DDR Dataset         12522
DeepDRiD              500
EyePACS (Kaggle)    17563
IDRiD                 516
Messidor-2            711
Name: patient_id, dtype: int64

Tổng số bệnh nhân duy nhất toàn bộ cohort: 35474


,image_id,diagnosis,dataset_name,dataset_key,image_file,raw_full_path,patient_id,side
0,1ae8c165fd53,2,APTOS 2019,APTOS,1ae8c165fd53.png,d:\Github\deep-learning-for-computer-vision\da...,APTOS::1ae8c165fd53,unknown
1,1b329a127307,1,APTOS 2019,APTOS,1b329a127307.png,d:\Github\deep-learning-for-computer-vision\da...,APTOS::1b329a127307,unknown
2,1b32e1d775ea,4,APTOS 2019,APTOS,1b32e1d775ea.png,d:\Github\deep-learning-for-computer-vision\da...,APTOS::1b32e1d775ea,unknown
3,1b3647865779,0,APTOS 2019,APTOS,1b3647865779.png,d:\Github\deep-learning-for-computer-vision\da...,APTOS::1b3647865779,unknown
4,1b398c0494d1,0,APTOS 2019,APTOS,1b398c0494d1.png,d:\Github\deep-learning-for-computer-vision\da...,APTOS::1b398c0494d1,unknown
...,...,...,...,...,...,...,...,...
55565,44347_right,0,EyePACS (Kaggle),EyePACS,44347_right.jpeg,d:\Github\deep-learning-for-computer-vision\da...,EyePACS::44347,right
55566,44348_left,0,EyePACS (Kaggle),EyePACS,44348_left.jpeg,d:\Github\deep-learning-for-computer-vision\da...,EyePACS::44348,left
55567,44348_right,0,EyePACS (Kaggle),EyePACS,44348_right.jpeg,d:\Github\deep-learning-for-computer-vision\da...,EyePACS::44348,right
55568,44349_left,0,EyePACS (Kaggle),EyePACS,44349_left.jpeg,d:\Github\deep-learning-for-computer-vision\da...,EyePACS::44349,left


## 4. Step 2 & Step 3: Geometric Preprocessing & Image Enhancement

**Auto-Crop ROI:** Thresholding technique to remove non-diagnostic black borders around the fundus circular region.  


In [31]:
def crop_fundus_roi(img_bgr: np.ndarray, tolerance: int = 7) -> np.ndarray:
    """
    Auto-crops uninformative black borders around fundus photograph.
    """
    if img_bgr is None or img_bgr.size == 0:
        return img_bgr
        
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mask = gray > tolerance
    
    if not np.any(mask):
        return img_bgr
        
    # Crop to bounding box of non-black pixels
    row_mask = np.any(mask, axis=1)
    col_mask = np.any(mask, axis=0)
    ymin, ymax = np.where(row_mask)[0][[0, -1]]
    xmin, xmax = np.where(col_mask)[0][[0, -1]]
    
    cropped = img_bgr[ymin:ymax+1, xmin:xmax+1]
    return cropped if cropped.size > 0 else img_bgr
def process_single_image(args):
    raw_path, out_path = args
    if os.path.exists(out_path):
        return True
    try:
        raw_img = cv2.imread(raw_path)
        if raw_img is None:
            return False
        cropped = crop_fundus_roi(raw_img)
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        # Lưu định dạng JPEG chất lượng cao 95 giúp encode nhanh và tối ưu I/O
        cv2.imwrite(out_path, cropped, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
        return True
    except Exception:
        return False

## 5. Step 4: Albumentations Augmentation & Targeted Training Set Balancing

**Balancing Strategy (Applied STRICTLY to 70% Training Set):**
- Target Count: Exactly 2,000 images per DR severity grade (10,000 total training images).
- Class 0, Class 1, Class 2: Randomly undersampled to 2,000 images.
- Class 3, Class 4: Oversampled & augmented via Albumentations to 2,000 images.
- Validation (8,097) & Test (8,097) sets: Left untouched in raw distribution.



In [32]:
# 5. TẦNG 1: XỬ LÝ VÀ LƯU ẢNH THEO DATASET (ĐỘC LẬP VỚI SPLIT)
print("\n>>> TẦNG 1: Xử lý và lưu kho ảnh Native Cropped độc lập...")

# 1. Tạo đường dẫn đích chuẩn hóa cho từng ảnh
master_df["native_path"] = master_df.apply(
    lambda r: os.path.join(
        NATIVE_PROCESSED_DIR, 
        r["dataset_key"], 
        f"{re.sub(r'[^\w\.-]', '_', str(r['image_id']))}.jpg"
    ),
    axis=1
)

# 2. Tạo sẵn thư mục con cho từng dataset để tránh xung đột I/O
for key in master_df["dataset_key"].unique():
    os.makedirs(os.path.join(NATIVE_PROCESSED_DIR, key), exist_ok=True)

# 3. Tạo danh sách các cặp đường dẫn (raw_path, native_path)
tasks = list(zip(master_df["raw_full_path"], master_df["native_path"]))

# 4. Giới hạn số worker an toàn (tối đa 8 luồng để tránh nghẽn I/O và tràn RAM)
num_workers = min(8, os.cpu_count() or 4)
print(f"[INFO] Khởi chạy xử lý đa luồng với {num_workers} workers...")

with ThreadPoolExecutor(max_workers=num_workers) as executor:
    results = list(
        tqdm(
            executor.map(process_single_image, tasks),
            total=len(tasks),
            desc="Processing Native Images"
        )
    )

# 5. Lọc lại các ảnh thực sự tồn tại thành công trên ổ cứng
master_df = master_df[master_df["native_path"].apply(os.path.exists)].reset_index(drop=True)
print(f" Hoàn tất Tầng 1: Đã lưu {len(master_df):,} ảnh vào kho Native Cropped.")


>>> TẦNG 1: Xử lý và lưu kho ảnh Native Cropped độc lập...
[INFO] Khởi chạy xử lý đa luồng với 8 workers...


Processing Native Images: 100%|██████████| 55570/55570 [11:09<00:00, 83.04it/s]  


 Hoàn tất Tầng 1: Đã lưu 54,883 ảnh vào kho Native Cropped.


In [33]:
# ==========================================
# 2. LẤY MẪU ĐẢM BẢO TỪNG DATASET KHÔNG BỊ BỎ SÓT (GUARANTEED MULTI-SOURCE SAMPLING)
# ==========================================
def create_balanced_train_pool(train_df: pd.DataFrame, target_range=(1500, 2500), seed=42) -> pd.DataFrame:
    balanced_dfs = []
    for grade in range(5):
        grade_subset = train_df[train_df["diagnosis"] == grade].copy().reset_index(drop=True)
        current_count = len(grade_subset)
        if current_count == 0:
            continue
            
        datasets_in_grade = grade_subset["dataset_name"].unique()
        num_datasets = len(datasets_in_grade)
        
        if current_count > target_range[1]:
            target_count = int(np.random.RandomState(seed + grade).randint(target_range[0], target_range[1] + 1))
            base_quota = target_count // num_datasets
            sampled_datasets = []
            leftover_quota = 0
            eligible = []
            
            for ds_name in datasets_in_grade:
                ds_subset = grade_subset[grade_subset["dataset_name"] == ds_name]
                if len(ds_subset) <= base_quota:
                    sampled_datasets.append(ds_subset)
                    leftover_quota += (base_quota - len(ds_subset))
                else:
                    eligible.append((ds_name, ds_subset, len(ds_subset)))
                    
            if eligible:
                quota_large = base_quota + (leftover_quota // len(eligible))
                for ds_name, ds_subset, count in eligible:
                    sampled_datasets.append(ds_subset.sample(min(quota_large, count), random_state=seed))
                    
            selected = pd.concat(sampled_datasets, ignore_index=True)
            if len(selected) > target_count:
                selected = selected.sample(target_count, random_state=seed).reset_index(drop=True)
            elif len(selected) < target_count:
                rem = grade_subset[~grade_subset["native_path"].isin(selected["native_path"])]
                if not rem.empty:
                    extra = rem.sample(min(target_count - len(selected), len(rem)), random_state=seed)
                    selected = pd.concat([selected, extra], ignore_index=True)
            balanced_dfs.append(selected)
        else:
            balanced_dfs.append(grade_subset)
            
    return pd.concat(balanced_dfs, ignore_index=True)
def split_master_pool_patient_level(df: pd.DataFrame, random_state=42):
    gss_train = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=random_state)
    train_idx, temp_idx = next(gss_train.split(df, groups=df['patient_id']))
    
    train_df = df.iloc[train_idx].copy().reset_index(drop=True)
    temp_df = df.iloc[temp_idx].copy().reset_index(drop=True)
    
    gss_test = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=random_state)
    val_idx, test_idx = next(gss_test.split(temp_df, groups=temp_df['patient_id']))
    
    val_df = temp_df.iloc[val_idx].copy().reset_index(drop=True)
    test_df = temp_df.iloc[test_idx].copy().reset_index(drop=True)
    
    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"
    return train_df, val_df, test_df

# ==========================================
# 5. THỰC THI TOÀN BỘ QUY TRÌNH
# ==========================================
# THỨ TỰ ĐÚNG: Chia split trên master_df ĐẦY ĐỦ trước -> Chỉ cân bằng trên tập TRAIN
print(">>> BƯỚC 1: Phân chia 70/15/15 ở cấp độ bệnh nhân trên toàn bộ Master DF (Chống Leakage)...")
train_raw_df, val_raw_df, test_raw_df = split_master_pool_patient_level(master_df, random_state=SEED)

print("\n>>> BƯỚC 2: Cân bằng lớp và đa nguồn CHỈ ÁP DỤNG TRÊN TẬP TRAIN...")
train_balanced_df = create_balanced_train_pool(train_raw_df, target_range=(1500, 2500), seed=SEED)

val_final_df = val_raw_df.copy()
test_final_df = test_raw_df.copy()

# 7. XUẤT FILE METADATA CSV (TỔNG KẾT)
train_balanced_df.to_csv(os.path.join(HYBRID_RAW_REF, "train_metadata.csv"), index=False)
val_final_df.to_csv(os.path.join(HYBRID_RAW_REF, "val_metadata.csv"), index=False)
test_final_df.to_csv(os.path.join(HYBRID_RAW_REF, "test_metadata.csv"), index=False)

print("\n" + "="*50)
print("=== THỐNG KÊ DATASET SAU PIPELINE ===")
print("="*50)
print("Train Set (Balanced):\n", train_balanced_df["diagnosis"].value_counts().sort_index())
print("\nVal Set (Natural Prevalence):\n", val_final_df["diagnosis"].value_counts().sort_index())
print("\nTest Set (Natural Prevalence):\n", test_final_df["diagnosis"].value_counts().sort_index())
print("\nTest Prevalence (%):\n", (test_final_df["diagnosis"].value_counts(normalize=True).sort_index() * 100).round(2))


>>> BƯỚC 1: Phân chia 70/15/15 ở cấp độ bệnh nhân trên toàn bộ Master DF (Chống Leakage)...

>>> BƯỚC 2: Cân bằng lớp và đa nguồn CHỈ ÁP DỤNG TRÊN TẬP TRAIN...

=== THỐNG KÊ DATASET SAU PIPELINE ===
Train Set (Balanced):
 diagnosis
0    1602
1    2336
2    2288
3    1326
4    1527
Name: count, dtype: int64

Val Set (Natural Prevalence):
 diagnosis
0    5286
1     550
2    1809
3     283
4     310
Name: count, dtype: int64

Test Set (Natural Prevalence):
 diagnosis
0    5176
1     603
2    1705
3     249
4     342
Name: count, dtype: int64

Test Prevalence (%):
 diagnosis
0    64.10
1     7.47
2    21.11
3     3.08
4     4.24
Name: proportion, dtype: float64


## 6. Step 6: Pixel Normalization & Pipeline Verification



In [ ]:
def normalize_image_tensor(img_np: np.ndarray, mode: str = "zscore") -> np.ndarray:
    """
    Normalizes pixel values to [0, 1] or standard ImageNet Z-score.
    """
    img_float = img_np.astype(np.float32) / 255.0
    if mode == "minmax":
        return img_float
    elif mode == "zscore":
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        return (img_float - mean) / std
    return img_float

# Save processed metadata tables
train_balanced_df.to_csv(os.path.join(BASE_RAW_DIR, "hybrid", "train_balanced_metadata.csv"), index=False)
val_final_df.to_csv(os.path.join(BASE_RAW_DIR, "hybrid", "val_raw_metadata.csv"), index=False)
test_final_df.to_csv(os.path.join(BASE_RAW_DIR, "hybrid", "test_raw_metadata.csv"), index=False)

print("\n=== FINAL PREPROCESSING & BALANCING SUMMARY ===")
print("Balanced Train Set (512x512):")
print(train_balanced_df["diagnosis"].value_counts().sort_index())

print("\nUntouched Validation Set:")
print(val_final_df["diagnosis"].value_counts().sort_index())

print("\nUntouched Test Set:")
print(test_final_df["diagnosis"].value_counts().sort_index())



=== FINAL PREPROCESSING & BALANCING SUMMARY ===
Balanced Train Set (300x300 & 512x512):
diagnosis
0    1602
1    2336
2    2288
3    1326
4    1527
Name: count, dtype: int64

Untouched Validation Set:
diagnosis
0    5286
1     550
2    1809
3     283
4     310
Name: count, dtype: int64

Untouched Test Set:
diagnosis
0    5176
1     603
2    1705
3     249
4     342
Name: count, dtype: int64
